# Data Preprocessing

Load the local processed comfort study data, inspect its schema, and prepare features for modeling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

# Define paths relative to the notebook
data_dir = '../data/raw/'
processed_dir = '../data/processed/'

# 1. Load Data
df_room = pd.read_csv(f'{data_dir}1_room_measurements.csv')
df_occ = pd.read_csv(f'{data_dir}3_room_occupancy.csv')
df_comfort = pd.read_csv(f'{data_dir}4_comfort_perception.csv')

# 2. Time-Alignment
df_room['timestamp'] = pd.to_datetime(df_room['timestamp']).dt.round('5min')
df_occ['timestamp'] = pd.to_datetime(df_occ['timestamp']).dt.round('5min')
df_comfort['timestamp'] = pd.to_datetime(df_comfort['timestamp']).dt.round('5min')

df_room_agg = df_room.groupby(['room', 'timestamp']).mean(numeric_only=True).reset_index()
df_occ_agg = df_occ.groupby(['room', 'timestamp']).mean(numeric_only=True).reset_index()
df_comfort_agg = df_comfort.groupby(['room', 'timestamp'])[['comfortValue']].mean().reset_index()

# 3. Merge
df_merged = pd.merge(df_room_agg, df_occ_agg, on=['room', 'timestamp'], how='left')
df_merged = pd.merge(df_merged, df_comfort_agg, on=['room', 'timestamp'], how='left')
df_merged = df_merged.sort_values(by=['room', 'timestamp'])

# 4. Feature Engineering
df_merged['ASR'] = df_merged.groupby('room')['CO2'].diff() / 5.0
df_merged['date'] = df_merged['timestamp'].dt.date
df_merged['session_start'] = df_merged.groupby(['room', 'date'])['timestamp'].transform('min')
df_merged['elapsed_time_min'] = (df_merged['timestamp'] - df_merged['session_start']).dt.total_seconds() / 60.0
df_merged = df_merged.drop(columns=['date', 'session_start'])

# 5. Label Construction
df_merged['Comfort_Score'] = ((df_merged['comfortValue'] - 1) / 4) * 100

def determine_attention(co2):
    if pd.isna(co2): return np.nan
    elif co2 >= 1000: return 'Low'
    elif co2 >= 800: return 'Medium'
    else: return 'High'

df_merged['Attention_Level'] = df_merged['CO2'].apply(determine_attention)

# 6. Cleaning and Scaling
cols_to_fill = ['temperature', 'humidity', 'noise', 'CO2', 'ASR', 'peopleCount', 'Comfort_Score', 'Attention_Level']
df_merged[cols_to_fill] = df_merged.groupby('room')[cols_to_fill].ffill().bfill()
df_merged['peopleCount'] = df_merged['peopleCount'].fillna(0)
df_clean = df_merged.dropna(subset=['Comfort_Score', 'Attention_Level']).copy()

features = ['temperature', 'humidity', 'noise', 'CO2', 'ASR', 'elapsed_time_min', 'peopleCount']
scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])

# 7. Export Processed Data
os.makedirs(processed_dir, exist_ok=True)
df_clean.to_csv(f'{processed_dir}processed_study_comfort_data.csv', index=False)
print("Data successfully preprocessed and saved.")